# Notebook 05: Inference on New Data

This notebook demonstrates how to load the saved preprocessing artifacts, the `feature_engineering` module, and the final trained model to generate predictions on completely new, unseen loan applications.

This simulates the exact process that would run in a production API or batch scoring job.

In [1]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# Add project root to Python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MODEL_DIR = PROJECT_ROOT / "models"

from src.feature_engineering import (
    apply_feature_pipeline,
    load_zip3_artifact,
    TargetEncoder,
)

## 1. Load Artifacts

We need to load:
1. The `zip3` target encoder JSON (for portable inference)
2. The `preprocessing_pipeline.pkl` (contains the scikit-learn preprocessor and feature names)
3. The `final_model.pkl` (the actual XGBoost model object)

In [2]:
# 1. Load preprocessing artifacts
with open(MODEL_DIR / "preprocessing_pipeline.pkl", "rb") as f:
    prep_artifacts = pickle.load(f)

preprocessor = prep_artifacts["preprocessor"]
feature_names = prep_artifacts["feature_names"]
missing_flag_cols = prep_artifacts.get("missing_flag_cols", [])
zip3_encoder_artifact = prep_artifacts.get("zip3_encoder_artifact", None)
zip3_keepers = prep_artifacts.get("zip3_keepers", None)

# 2. Load final trained model
with open(MODEL_DIR / "final_model.pkl", "rb") as f:
    model = pickle.load(f)

print("Artifacts loaded successfully.")
print(f"Number of transformed feature names: {len(feature_names)}")
print(f"Missing-flag columns expected: {len(missing_flag_cols)}")

Artifacts loaded successfully.
Number of transformed feature names: 118
Missing-flag columns expected: 29


In [5]:
# Load zip3 encoder
ZIP3_JSON_PATH = MODEL_DIR / "zip3_encoder.json"

saved_zip3_encoder_artifact = prep_artifacts.get("zip3_encoder_artifact", None)
saved_zip3_keepers = prep_artifacts.get("zip3_keepers", None)

zip3_encoder = None
zip3_keepers = None

if ZIP3_JSON_PATH.exists():
    zip3_encoder, zip3_keepers = load_zip3_artifact(ZIP3_JSON_PATH)
    print("Loaded zip3 encoder from JSON.")

elif saved_zip3_encoder_artifact is not None:
    zip3_encoder = TargetEncoder.from_artifact(saved_zip3_encoder_artifact)
    zip3_keepers = set(saved_zip3_keepers) if saved_zip3_keepers is not None else None
    print("Loaded zip3 encoder from preprocessing artifacts.")

else:
    print("No zip3 encoder artifact found. zip3 will not be encoded.")

Loaded zip3 encoder from JSON.


## 2. Create a sample raw input

In [15]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "accepted_2007_to_2018Q4.csv.gz"

df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

# Keep only the same population used in training if needed
df_raw = df_raw[df_raw["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df_raw = df_raw[df_raw["application_type"] == "Individual"].copy()

# Take one real raw row
df_raw_input = df_raw.sample(1, random_state=42).copy()

# Drop target-related columns if present
cols_to_remove = ["loan_status", "target_default"]
df_raw_input = df_raw_input.drop(columns=[c for c in cols_to_remove if c in df_raw_input.columns])

print(df_raw_input.shape)
df_raw_input.head()

(1, 150)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
1807049,6886107,NaN,13000.0,13000.0,13000.0,36 months,15.22,452.06,C,C3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_fe = apply_feature_pipeline(
    df_raw_input,
    encode_zip3=(zip3_encoder is not None),
    zip3_encoder=zip3_encoder,
    zip3_keepers=zip3_keepers,
)

print("Feature-engineered shape:", df_fe.shape)
print("Columns after feature engineering:")
print(sorted(df_fe.columns))

Feature-engineered shape: (1, 136)
Columns after feature engineering:
['acc_now_delinq', 'acc_open_past_24mths', 'all_util', 'annual_inc', 'annual_inc_joint', 'application_type', 'avg_bal_per_account', 'bc_open_to_buy_ratio', 'chargeoff_within_12_mths', 'collection_recovery_fee', 'collections_12_mths_ex_med', 'credit_age_months', 'debt_settlement_flag', 'debt_settlement_flag_date', 'deferral_term', 'delinq_2yrs', 'delinq_amnt', 'delinq_ratio', 'desc', 'dti', 'dti_joint', 'emp_length', 'emp_title', 'fico_avg', 'hardship_amount', 'hardship_dpd', 'hardship_end_date', 'hardship_flag', 'hardship_last_payment_amount', 'hardship_length', 'hardship_loan_status', 'hardship_payoff_balance_amount', 'hardship_reason', 'hardship_start_date', 'hardship_status', 'hardship_type', 'home_ownership', 'id', 'il_util', 'initial_list_status', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment', 'installment_to_income', 'int_rate', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'l

/Users/talia/Desktop/MSFT/Job Hunt/LendingClub_Issued_loans/github_portfolio/credit-risk-default-prediction/src/feature_engineering.py:508: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace([np.inf, -np.inf], np.nan)


In [24]:
# Recreate missing-indicator columns expected by the preprocessor
df_model_input = df_fe.copy()

for flag_col in missing_flag_cols:
    if flag_col not in df_model_input.columns:
        base_col = flag_col.replace("_missing", "")
        if base_col in df_model_input.columns:
            df_model_input[flag_col] = df_model_input[base_col].isna().astype(int)
        else:
            df_model_input[flag_col] = 1

print("Shape after adding missing flags:", df_model_input.shape)

Shape after adding missing flags: (1, 165)


In [25]:
required_input_cols = list(preprocessor.feature_names_in_)

missing_required_cols = [c for c in required_input_cols if c not in df_model_input.columns]
extra_cols = [c for c in df_model_input.columns if c not in required_input_cols]

print("Missing required columns:", missing_required_cols)
print("Extra columns not used by preprocessor:", extra_cols)

# Add missing required columns as NaN
for col in missing_required_cols:
    df_model_input[col] = np.nan

# Keep only the columns expected by the preprocessor, in the right order
df_model_input = df_model_input[required_input_cols]

print("Final model-input shape:", df_model_input.shape)
df_model_input.head()

Missing required columns: []
Extra columns not used by preprocessor: ['id', 'member_id', 'emp_title', 'pymnt_plan', 'url', 'desc', 'title', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 

,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,...,num_actv_rev_tl_missing,num_il_tl_missing,num_tl_120dpd_2m_missing,num_tl_90g_dpd_24m_missing,num_tl_op_past_12m_missing,pct_tl_nvr_dlq_missing,revol_bal_to_limit_missing,avg_bal_per_account_missing,delinq_ratio_missing,zip3_encoded
1807049,13000.0,36 months,15.22,452.06,C3,Unknown,RENT,49000.0,Verified,credit_card,...,0,0,0,0,0,0,0,0,0,0.211125


## 3. Apply the fitted preprocessor

In [26]:
X_input = preprocessor.transform(df_model_input)

print("Transformed input shape:", X_input.shape)

Transformed input shape: (1, 118)


In [30]:
with open(MODEL_DIR / "best_model_metadata.pkl", "rb") as f:
    best_meta = pickle.load(f)

best_threshold = best_meta["best_threshold"]

y_prob = model.predict_proba(X_input)[:, 1][0]
y_pred = int(y_prob >= best_threshold)

print(f"Predicted default probability: {y_prob:.4f}")
print(f"Predicted class at threshold {best_threshold:.2f}: {y_pred}")

Predicted default probability: 0.6310
Predicted class at threshold 0.54: 1


In [32]:
result = pd.DataFrame({
    "predicted_default_probability": [y_prob],
    "predicted_class_best_threshold": [y_pred]
})

result

,predicted_default_probability,predicted_class_best_threshold
0,0.63097,1
